#**1. Data Quality Assesment**

**1.1 Data set overview**

In [ ]:
# import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# load dataset
df_raw = pd.read_csv('/content/dirty_cafe_sales.csv')

# melakukan copy untuk preprocessing
df_prep = df_raw.copy()

# preview data
df_prep.head(20)



In [ ]:
# melihat ukuran dataset
df_prep.shape

In [ ]:
# melihat struktur dan tipe data
df_prep.info()

In [ ]:
# melihat statistiknya
df_prep.describe().T

# **2. Data quality issues**

In [ ]:
# melakukan cek missing values
pd.DataFrame({
    'Missing values': df_prep.isnull().sum(),
    'Missing percentage (%)': df_prep.isnull().sum() / len(df_prep)*100
})

In [ ]:
# cek duplikasi

df_prep.duplicated().sum()

In [ ]:
# cek tipe data
df_prep.dtypes

In [ ]:
# cek nilai tidak valid, seperti error atau unknown
for col in df_prep.columns:
  print(f"\n{col}")
  print(df_prep[col].unique())

In [ ]:
# melakukan cek deskripsi data
df_prep.describe().T

# **3. Data Wrangling & Preparation**

**3.1 Data cleaning steps**

In [ ]:
# menghapus data yang terduplikasi
df_prep = df_prep.drop_duplicates().reset_index(drop = True)

In [ ]:
# mengubah nilai yang tidak valid, seperti error/uknown, jadi NaN
df_prep = df_prep.replace(['UNKNOWN', 'ERROR'], pd.NA)

In [ ]:
# mengubah isi kolom diubah menjadi angka dan tanggal
df_prep['Quantity'] = pd.to_numeric(df_prep['Quantity'], errors = 'coerce')
df_prep['Price Per Unit'] = pd.to_numeric(df_prep['Price Per Unit'], errors = 'coerce')
df_prep['Total Spent'] = pd.to_numeric(df_prep['Total Spent'], errors = 'coerce')
df_prep['Transaction Date'] = pd.to_datetime(df_prep['Transaction Date'], errors = 'coerce')

# cek tipe data
df_prep.dtypes

In [ ]:
# menghitung total spent untuk mengisi kolom yang kosong saja
df_prep['Total Spent'] = df_prep['Total Spent'].fillna(df_prep['Quantity'] * df_prep['Price Per Unit'])

In [ ]:
# menghapus baris yang memiliki nilai kosong atau NaN pada dikolom yang disebutkan
df_prep = df_prep.dropna(subset=['Quantity', 'Price Per Unit', 'Total Spent', 'Transaction Date', 'Item'])

In [ ]:
# mengubah tipe data quantity menjadi lebih sesuai
df_prep['Quantity'] = df_prep['Quantity'].astype('int64')

In [ ]:
# melihat data setelah cleaning
df_prep.head(20)

In [ ]:
# melihat struktur dan tipe data yang sudah di cleaning
df_prep.info()

**3.2 Handling missing values & Outliers**

In [ ]:
# mengecek jumlah missing value
pd.DataFrame({
    'Missing values': df_prep.isnull().sum(),
    'Missing percentage (%)': df_prep.isnull().sum() / len(df_prep)*100
})

In [ ]:
# mengisi baris yang memiliki missing value dengan nilai yang paling sering muncul
df_prep['Payment Method'] = df_prep['Payment Method'].fillna(df_prep['Payment Method'].mode()[0])
df_prep['Location'] = df_prep['Location'].fillna(df_prep['Location'].mode()[0])

In [ ]:
# mengecek berapa persen missing value yang ada
pd.DataFrame({
    'Missing Count' : df_prep.isnull().sum(),
    'Missing Precentage' : df_prep.isnull().sum() / len(df_prep) * 100
})

In [ ]:
# melakukan penyimpanan kondisi awal sebelum mendeteksi dan menangani outliers yang ada
df_before = df_prep.copy()

print("Before:", len(df_before))

In [ ]:
# memilih kolom yang berupa angka
numerical_cols = df_prep.select_dtypes(include=['float64', 'int64']).columns

print("--- BATAS AMAN OUTLIER (METODE IQR) ---")
# Loop untuk melihat range batas bawah dan atas sebelum dihapus
for col in numerical_cols:
    Q1 = df_prep[col].quantile(0.25)
    Q3 = df_prep[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    lower_adjusted = max(0, lower)

    print(f"Kolom '{col}':")
    print(f"  - Batas Minimum Aman (Lower): {lower}")
    print(f"  - Batas Maksimum Aman (Upper): {upper}")
    print(f"  - Data di luar range ini akan dianggap OUTLIER.\n")

In [ ]:
# memilih kolom yang berupa angka
numerical_cols =  df_prep.select_dtypes(include=['float64']).columns

# mengahapus outliers yang ada
for col in numerical_cols:
    Q1 = df_prep[col].quantile(0.25)
    Q3 = df_prep[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df_prep = df_prep[(df_prep[col] >= lower) & (df_prep[col] <= upper)]

df_prep = df_prep.reset_index(drop=True)

# hasil outliers yang terhapus
print("After:", len(df_prep))


In [ ]:
# mengecek outliers yang sudah dicleaning
df_prep.shape

In [ ]:
for col in df_prep.columns:
  print(f"\n{col}")
  print(df_prep[col].unique())

In [ ]:
# Cek data yang masih kosong (NaN)
print(df_prep.isnull().sum())

In [ ]:
# melihat struktur dan tipe data yang sudah di cleaning
df_prep.info()

In [ ]:
# preview data yang sudah di cleaning
df_prep.head(20)

**3.3 Data transformation & Derived Variables**

In [ ]:
# mencari nama bulan dari transaction date
df_prep['Month'] = df_prep['Transaction Date'].dt.month_name()

In [ ]:
# membagi kategori hari weekday dan weekend
df_prep['Day Type'] = df_prep['Transaction Date'].dt.day_name().apply(lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday')
df_prep['Day Name'] = df_prep['Transaction Date'].dt.day_name()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_prep['Day Name'] = pd.Categorical(df_prep['Day Name'], categories=days_order, ordered=True)

In [ ]:
# membagi kategori lokasi order
df_prep['Order Status'] = df_prep['Location'].replace({
    'In-Store' : 'Dine-In',
    'Takeaway' : 'To-Go',
})

In [ ]:
# hasil
df_prep[['Transaction Date', 'Month', 'Day Type', 'Order Status', 'Day Name']].head(25)

In [ ]:
# Export data yang sudah clean
df_prep.to_csv('cleaned_cafe_sales.csv', index=False)

# **4. Exploratory Data Analysis**

**4.1 Present key descriptive statistics**

In [ ]:
# melihat statistik setelah cleaning
df_prep.describe().T

**4.2 Visualize the data**

In [ ]:
# distribution histogram sebaran uang customer
plt.figure(figsize=(10, 5))
sns.histplot(df_prep['Total Spent'], kde=True, color='teal', bins=20)
plt.title('Distribution of Customer Spending', fontsize=14)
plt.xlabel('Total Spent')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# menghitung total revenue per bulan
months_order = ['January', 'February', 'March', 'April', 'May', 'June',
                'July', 'August', 'September', 'October', 'November', 'December']

monthly_revenue = df_prep.groupby('Month')['Total Spent'].sum().reindex(months_order)

plt.figure(figsize=(12, 5))
plt.plot(monthly_revenue.index, monthly_revenue.values, marker='o', color='steelblue', linewidth=2)
plt.title('Monthly Revenue Trend', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Total Revenue')
plt.ylim(0, monthly_revenue.max() * 1.2)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# menghitung total revenue per item
revenue_per_item = df_prep.groupby('Item')['Total Spent'].sum().sort_values(ascending=False)
colors = ['orange' if x == revenue_per_item.max() else 'skyblue' for x in revenue_per_item.values]

plt.figure(figsize=(10, 5))
sns.barplot(x=revenue_per_item.index, y=revenue_per_item.values, palette=colors, hue=revenue_per_item.index, legend=False)
plt.title('Total Revenue per Item', fontsize=14)
plt.xlabel('Item')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Bar Chart, comparisons produk paling laku
plt.figure(figsize=(10, 5))
item_counts = df_prep['Item'].value_counts()
colors = ['yellow' if (x == item_counts.max()) else 'skyblue' for x in item_counts.values]
sns.barplot(x=item_counts.index, y=item_counts.values, palette=colors, hue=item_counts.index, legend=False)
plt.title('Top Selling Items', fontsize=12)
plt.xticks(rotation=45)
plt.ylabel('Quantity Sold')
plt.show()


In [ ]:
# membandingkan penggunaan jenis pembayaran

plt.figure(figsize=(7, 7))
day_counts = df_prep['Payment Method'].value_counts()
plt.pie(day_counts, labels = day_counts.index, autopct = '%1.1f%%', colors = ['skyblue', 'yellow','pink'], startangle = 90)
plt.title('Payment Method Proporsion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# menghitung jumlah transaksi per hari
plt.figure(figsize=(10, 6))
day_counts = df_prep['Day Name'].value_counts().reindex(days_order)
colors = ['orange' if (x == day_counts.max()) else 'skyblue' for x in day_counts]
sns.barplot(x=day_counts.index, y=day_counts.values, palette=colors, hue=day_counts.index, legend=False)
plt.title('Number of Transactions by Day', fontsize=14)
plt.xlabel('Day')
plt.ylabel('Number Of Transaction')
plt.xticks(rotation=45)
plt.show()

# **5. Advanced Feature Engineering**

In [ ]:
# load dataset
df_clean = pd.read_csv('/content/cleaned_cafe_sales.csv')

# melakukan copy untuk preprocessing
df_clean = df_clean.copy()

# preview data
df_clean.head(20)

In [ ]:
# Memastikan nilai minimum dan maksimum masih masuk akal

df_clean[
    ['Quantity', 'Price Per Unit', 'Total Spent']
].agg(['min', 'max'])

In [ ]:
# Menghitung jumlah transaksi yang valid dan tidak valid

check_total = pd.Series(
    np.isclose(
        df_clean['Total Spent'],
        df_clean['Quantity'] * df_clean['Price Per Unit']
    )
)

pd.DataFrame({
    'Jumlah': check_total.value_counts(),
    'Persentase (%)': check_total.value_counts(normalize=True) * 100
}).round(2)

## 5.1. Product Category Engineering


In [ ]:
# Product Category

df_analysis = df_clean.copy()

beverage_items = ['Coffee', 'Tea', 'Juice', 'Smoothie']
food_items = ['Cake', 'Cookie', 'Sandwich', 'Salad']

df_analysis['Product Category'] = df_analysis['Item'].apply(
    lambda x: 'Beverage' if x in beverage_items else 'Food'
)
counts = df_analysis['Product Category'].value_counts()

percentages = df_analysis['Product Category'].value_counts(normalize=True) * 100

df_summary = pd.DataFrame({
    'Count': counts,
    'Percentage (%)': percentages.round(2)
})

df_summary

In [ ]:
plt.figure(figsize=(10, 6))

category_counts = df_analysis['Product Category'].value_counts()

sns.barplot(
    x=category_counts.index,
    y=category_counts.values,
    legend=False
)

plt.title('Product Category Distribution', fontsize=14)
plt.xlabel('Product Category')
plt.ylabel('Number of Transactions')

plt.show()



## 5.2. Purchase Size Engineering

In [ ]:
df_analysis['Purchase Size'] = pd.cut(
    df_analysis['Quantity'],
    bins=[0,2,4,5],
    labels=['Small','Medium','Large'],
    include_lowest=True
)
counts = df_analysis['Purchase Size'].value_counts()

percentages = df_analysis['Purchase Size'].value_counts(normalize=True) * 100

df_summary = pd.DataFrame({
    'Count': counts,
    'Percentage (%)': percentages.round(2)
})

df_summary

In [ ]:
plt.figure(figsize=(10, 6))

purchase_counts = df_analysis['Purchase Size'].value_counts().reindex(
    ['Small','Medium','Large']
)

sns.barplot(
    x=purchase_counts.index,
    y=purchase_counts.values,
    legend=False
)

plt.title('Purchase Size Distribution', fontsize=14)
plt.xlabel('Purchase Size')
plt.ylabel('Number of Transactions')

plt.show()

##5.3 Spending Level Engineering

In [ ]:
# Spending Level menggunakan threshold tetap (fixed threshold)
def get_spending_level(spent):
    if spent <= 5.0:
        return 'Budget'
    elif spent <= 10.0:
        return 'Standard'
    elif spent <= 15.0:
        return 'Premium'
    else:
        return 'Luxury'

df_analysis['Spending Level'] = df_analysis['Total Spent'].apply(get_spending_level)

counts = df_analysis['Spending Level'].value_counts()

percentages = df_analysis['Spending Level'].value_counts(normalize=True) * 100

df_summary = pd.DataFrame({
    'Count': counts,
    'Percentage (%)': percentages.round(2)
})

df_summary

In [ ]:
plt.figure(figsize=(10, 6))

spending_counts = df_analysis['Spending Level'].value_counts().reindex(
    ['Budget','Standard','Premium','Luxury']
)

sns.barplot(
    x=spending_counts.index,
    y=spending_counts.values,
    legend=False
)

plt.title('Spending Level Distribution', fontsize=14)
plt.xlabel('Spending Level')
plt.ylabel('Number of Transactions')

plt.show()

#**6. Exporting Engineered Dataset**

In [ ]:
# Check kolom yang baru ditambahkan kedalam dataset
df_analysis.head(25)

In [ ]:
# melihat struktur dan tipe data yang di feature engineer
df_analysis.info()

In [ ]:
# melihat hasil statistiknya
df_analysis.describe().T

In [ ]:
# export data yang sudah ditambahkan kolom feature engineernya
df_analysis.to_csv(
    'cafe_sales_analysis.csv',
    index=False
)

# **7.Relationship Analysis**

## 7.1. Product Category vs Order Status

In [ ]:
#Crosstab
category_status = pd.crosstab(
    df_analysis['Product Category'],
    df_analysis['Location']
)

category_status

In [ ]:
#Presentase
category_status_pct = pd.crosstab(
    df_analysis['Product Category'],
    df_analysis['Location'],
    normalize='index'
) * 100

category_status_pct.round(2)

In [ ]:
#Visualisasi
category_status_pct.plot(
    kind='bar',
    figsize=(8,5)
)

plt.title('Product Category vs Order Status')
plt.xlabel('Product Category')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Order Status')
plt.show()

## 7.2 Product Category vs Payment Method

In [ ]:
#Crosstab
category_payment = pd.crosstab(
    df_analysis['Product Category'],
    df_analysis['Payment Method']
)

category_payment

In [ ]:
#Persentase
category_payment_pct = pd.crosstab(
    df_analysis['Product Category'],
    df_analysis['Payment Method'],
    normalize='index'
) * 100

category_payment_pct.round(2)

In [ ]:
#Visualisasi
category_payment_pct.plot(
    kind='bar',
    figsize=(8,5)
)

plt.title('Product Category vs Payment Method')
plt.xlabel('Product Category')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Payment Method')
plt.show()

## 7.3 Weekday vs Weekend Analysis

In [ ]:
#Membuat ketgori hari
df_analysis['Transaction Date'] = pd.to_datetime(
    df_analysis['Transaction Date']
)

df_analysis['Day Type'] = np.where(
    df_analysis['Transaction Date'].dt.dayofweek >= 5,
    'Weekend',
    'Weekday'
)

In [ ]:
#Crosstab Product Category vs Day Type
weekday_weekend = pd.crosstab(
    df_analysis['Day Type'],
    df_analysis['Product Category']
)

weekday_weekend

In [ ]:
#Persentase
weekday_weekend_pct = pd.crosstab(
    df_analysis['Day Type'],
    df_analysis['Product Category'],
    normalize='index'
) * 100

weekday_weekend_pct.round(2)

In [ ]:
#Visualisasi
weekday_weekend_pct.plot(
    kind='bar',
    figsize=(8,5)
)

plt.title('Weekday vs Weekend Product Category')
plt.xlabel('Day Type')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Product Category')
plt.show()

## 7.4 Purchase Size vs Order Status

In [ ]:
def get_purchase_size(qty):
    if qty <= 2:
        return 'Small'
    elif qty <= 4:
        return 'Medium'
    else:
        return 'Large'

df_analysis['Purchase Size'] = df_analysis['Quantity'].apply(get_purchase_size)

In [ ]:
def get_spending_level(spent):
    if spent <= 5.0:
        return 'Budget'
    elif spent <= 10.0:
        return 'Standard'
    elif spent <= 15.0:
        return 'Premium'
    else:
        return 'Luxury'

df_analysis['Spending Level'] = df_analysis['Total Spent'].apply(get_spending_level)

In [ ]:
def get_product_category(item):
    beverages = ['Coffee', 'Smoothie', 'Juice', 'Tea']
    if item in beverages:
        return 'Beverage'
    else:
        return 'Food'

df_analysis['Product Category'] = df_analysis['Item'].apply(get_product_category)

print("Kolom 'Purchase Size', 'Spending Level', dan 'Product Category' BERHASIL DIBUAT!")

In [ ]:
# Crosstab
crosstab_63 = pd.crosstab(df_analysis['Purchase Size'],
                          df_analysis['Order Status'])
print("\n[Crosstab Frequencies]")
crosstab_63

In [ ]:
# presentase proporsi perbaris/purchaase size
percentage_63 = pd.crosstab(df_analysis['Purchase Size'], df_analysis['Order Status'], normalize='index') * 100
print("\n[Persentase Kontingensi (%)]")
percentage_63.round(2)

In [ ]:
# Visualisasi
plt.figure(figsize=(10, 6))
percentage_63.plot(kind='bar', stacked=True, color=['#4caf50', '#ff9800'], ax=plt.gca())
plt.title('Analisis Purchase Size vs Order Status (Persentase)', fontsize=14)
plt.xlabel('Purchase Size')
plt.ylabel('Persentase (%)')
plt.legend(title='Order Status')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7.5 Spending Level vs Product Category

In [ ]:
import pandas as pd

# Menggunakan normalize='all' untuk persentase dari total keseluruhan data
crosstab_65_all = pd.crosstab(
    df_analysis['Spending Level'],
    df_analysis['Product Category'],
    normalize='all'
) * 100

display(crosstab_65_all.round(2))

In [ ]:
# Crosstab
crosstab_65 = pd.crosstab(df_analysis['Spending Level'],
                          df_analysis['Product Category'])
print("\n[Crosstab Frekuensi]")
crosstab_65.round(2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Menghitung crosstab dalam bentuk persentase dari TOTAL KESELURUHAN DATA
crosstab_65_pct = pd.crosstab(
    df_analysis['Spending Level'],
    df_analysis['Product Category'],
    normalize='all'
) * 100

print("\n[Tabel Crosstab Persentase - Dari Total Semua Data]")
display(crosstab_65_pct.round(2))
print("-" * 50)

# 2. Pembuatan Visualisasi Grafik Batang
plt.figure(figsize=(10, 6))
crosstab_65_pct.plot(kind='bar', color=['#2196f3', '#ff5722'], ax=plt.gca())

plt.title('Dominasi Produk pada Masing-Masing Level Pengeluaran (Persentase Total)', fontsize=14)
plt.xlabel('Spending Level')
plt.ylabel('Persentase (%)')
plt.legend(title='Product Category')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi
plt.figure(figsize=(10, 6))
crosstab_65.plot(kind='bar', color=['#2196f3', '#ff5722'], ax=plt.gca())
plt.title('Dominasi Produk pada Masing-Masing Level Pengeluaran', fontsize=14)
plt.xlabel('Spending Level')
plt.ylabel('Jumlah Pembelian')
plt.legend(title='Product Category')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 7.6 Product Category × Order Status × Day Type

In [ ]:
# Menghitung jumlah asli (Count) menggunakan crosstab
multi_1 = pd.crosstab(
    [df_analysis['Product Category'],
     df_analysis['Day Type']],
    df_analysis['Order Status']
)

# Reset index agar kolomnya rapi dan bisa difilter
multi_1 = multi_1.reset_index()

# Memisahkan data berdasarkan Day Type
weekday_data = multi_1[multi_1['Day Type'] == 'Weekday']
weekend_data = multi_1[multi_1['Day Type'] == 'Weekend']

# Menampilkan hasil jumlah transaksi asli
display(weekday_data)

display(weekend_data)

In [ ]:
# Crosstab
multi_1 = pd.crosstab(
    [df_analysis['Product Category'],
     df_analysis['Day Type']],
    df_analysis['Order Status'],
    normalize='index'
) * 100

multi_1 = multi_1.reset_index()

# Dipisahkan berdasarkan Day Type
weekday_data = multi_1[
    multi_1['Day Type'] == 'Weekday'
]

weekend_data = multi_1[
    multi_1['Day Type'] == 'Weekend'
]

display(weekday_data.round(2))
print()
display(weekend_data.round(2))

In [ ]:
# Visualisasi
fig, axes = plt.subplots(
    1, 2,
    figsize=(12,5),
    sharey=True
)

# WEEKDAY
weekday_data.set_index('Product Category')[
    ['In-store', 'To-Go']
].plot(
    kind='bar',
    color=['#2196f3', '#ff5722'],
    ax=axes[0]
)

axes[0].set_title('Weekday')
axes[0].set_xlabel('Product Category')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Order Status')
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# WEEKEND
weekend_data.set_index('Product Category')[
    ['In-store', 'To-Go']
].plot(
    kind='bar',
    color=['#2196f3', '#ff5722'],
    ax=axes[1]
)

axes[1].set_title('Weekend')
axes[1].set_xlabel('Product Category')
axes[1].set_ylabel('')
axes[1].legend(title='Order Status')
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.suptitle(
    'Order Status berdasarkan Product Category dan Day Type',
    fontsize=14
)

plt.tight_layout()
plt.show()

## 7.7 Purchase Size × Spending Level × Payment Method

In [ ]:
# Menghitung jumlah asli (Count) menggunakan crosstab
multi_2 = pd.crosstab(
    [df_analysis['Purchase Size'],
     df_analysis['Spending Level']],
    df_analysis['Payment Method']
)

# Reset index agar kolomnya rapi dan bisa difilter
multi_2 = multi_2.reset_index()

# Memisahkan data berdasarkan Purchase Size
small_data = multi_2[multi_2['Purchase Size'] == 'Small']
medium_data = multi_2[multi_2['Purchase Size'] == 'Medium']
large_data = multi_2[multi_2['Purchase Size'] == 'Large']

# Menampilkan hasil jumlah transaksi asli
print("TRANSAKSI UKURAN SMALL")
display(small_data)

print("\nTRANSAKSI UKURAN MEDIUM")
display(medium_data)

print("\nTRANSAKSI UKURAN LARGE")
display(large_data)

In [ ]:
# Crosstab
multi_2 = pd.crosstab(
    [df_analysis['Purchase Size'],
     df_analysis['Spending Level']],
    df_analysis['Payment Method'],
    normalize='index'
) * 100

multi_2 = multi_2.reset_index()

multi_2.round(2)

small_data = multi_2[
    multi_2['Purchase Size'] == 'Small'
]

medium_data = multi_2[
    multi_2['Purchase Size'] == 'Medium'
]

large_data = multi_2[
    multi_2['Purchase Size'] == 'Large'
]

display(small_data.round(2))
print()
display(medium_data.round(2))
print()
display(large_data.round(2))

In [ ]:
# Visualisasi
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18,5),
    sharey=True
)

# SMALL

small_data.set_index(
    'Spending Level'
)[['Cash','Credit Card','Digital Wallet']].plot(
    kind='bar',
    ax=axes[0]
)

axes[0].set_title('Small Purchase')
axes[0].set_xlabel('Spending Level')
axes[0].set_ylabel('Percentage (%)')
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# MEDIUM

medium_data.set_index(
    'Spending Level'
)[['Cash','Credit Card','Digital Wallet']].plot(
    kind='bar',
    ax=axes[1]
)

axes[1].set_title('Medium Purchase')
axes[1].set_xlabel('Spending Level')
axes[1].set_ylabel('')
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

# LARGE

large_data.set_index(
    'Spending Level'
)[['Cash','Credit Card','Digital Wallet']].plot(
    kind='bar',
    ax=axes[2]
)

axes[2].set_title('Large Purchase')
axes[2].set_xlabel('Spending Level')
axes[2].set_ylabel('')
axes[2].grid(axis='y', linestyle='--', alpha=0.7)

plt.suptitle(
    'Payment Method by Purchase Size and Spending Level',
    fontsize=14
)

plt.tight_layout()
plt.show()

## 7.8 Revenue Contribution Analysis

In [ ]:
# Mengukur kontribusi revenue berdasarkan kategori produk

rev_product = df_analysis.groupby(
    'Product Category'
)['Total Spent'].sum().reset_index()

total_rev = rev_product['Total Spent'].sum()

rev_product['Percentage (%)'] = (
    rev_product['Total Spent'] / total_rev
) * 100

print("Revenue Contribution by Product Category")
print()
print(rev_product.round(2))

In [ ]:
# Visualisasi kontribusi revenue masing-masing kategori produk

plt.figure(figsize=(5,5))

plt.pie(
    rev_product['Total Spent'],
    labels=rev_product['Product Category'],
    autopct='%1.1f%%',
    startangle=90
)

plt.title('Revenue Contribution by Product Category')

plt.show()

In [ ]:
# Mengukur kontribusi revenue berdasarkan ukuran pembelian

rev_size = df_analysis.groupby(
    'Purchase Size'
)['Total Spent'].sum().reset_index()

rev_size['Percentage (%)'] = (
    rev_size['Total Spent'] / total_rev
) * 100

print("Revenue Contribution by Purchase Size")
print()
print(rev_size.round(2))

In [ ]:
# Visualisasi kontribusi revenue masing-masing ukuran pembelian

plt.figure(figsize=(5,5))

plt.pie(
    rev_size['Total Spent'],
    labels=rev_size['Purchase Size'],
    autopct='%1.1f%%',
    startangle=90
)
# ERROR
plt.title('Revenue Contribution by Purchase Size')

plt.show()

In [ ]:
# Mengukur kontribusi revenue berdasarkan tingkat pengeluaran pelanggan

rev_spending = df_analysis.groupby(
    'Spending Level'
)['Total Spent'].sum().reset_index()

rev_spending['Percentage (%)'] = (
    rev_spending['Total Spent'] / total_rev
) * 100

print("Revenue Contribution by Spending Level")
print()
print(rev_spending.round(2))

In [ ]:
# Visualisasi kontribusi revenue pada setiap tingkat pengeluaran pelanggan

plt.figure(figsize=(5,5))

plt.pie(
    rev_spending['Total Spent'],
    labels=rev_spending['Spending Level'],
    autopct='%1.1f%%',
    startangle=90
)

plt.title('Revenue Contribution by Spending Level')

plt.show()

## 7.9 Statistical Validation using Chi-Square Test and Cramer's V

In [ ]:
def chi_square_test(df, row, col):

    contingency = pd.crosstab(df[row], df[col])

    chi2, p, dof, expected = chi2_contingency(contingency)

    n = contingency.values.sum()

    cramers_v = np.sqrt(
        chi2 / (n * (min(contingency.shape) - 1))
    )

    print("="*65)
    print(f"{row} vs {col}")
    print("="*65)

    display(contingency)

    print(f"\nChi-Square Statistic : {chi2:.4f}")
    print(f"P-value              : {p:.6f}")
    print(f"Degrees of Freedom   : {dof}")
    print(f"Cramer's V           : {cramers_v:.4f}")

    if p < 0.05:
        print("\nInterpretation : ")
        print("There is a statistically significant relationship.")
    else:
        print("\nInterpretation : ")
        print("There is no statistically significant relationship.")

    if cramers_v < 0.10:
        strength = "Negligible"

    elif cramers_v < 0.20:
        strength = "Weak"

    elif cramers_v < 0.40:
        strength = "Moderate"

    elif cramers_v < 0.60:
        strength = "Relatively Strong"

    else:
        strength = "Strong"

    print(f"Relationship Strength : {strength}")

    print("\n")

In [ ]:
chi_square_test(
    df_analysis,
    "Product Category",
    "Order Status"
)

In [ ]:
chi_square_test(
    df_analysis,
    "Product Category",
    "Payment Method"
)

In [ ]:
chi_square_test(
    df_analysis,
    "Spending Level",
    "Product Category"
)

# **8.Evaluation**

# 8.1 Code Evaluation

In [ ]:
# Cek Distribusi kolom kolom baru
target_cols = ['Product Category', 'Purchase Size', 'Spending Level']
for col in target_cols:
    counts = df_analysis[col].value_counts()
    pct = df_analysis[col].value_counts(normalize=True) * 100

    df_eval = pd.DataFrame({
        'Jumlah (Freq)': counts,
        'Persentase (%)': pct
    })
    print(f"\nDistribusi untuk kolom: {col}")
    print(df_eval.round(2))


    # PERHATIKAN  PURCHASE SIZE


8.2 Markdown Evaluation

Berdasarkan hasil dari code evaluation di atas dan setelah melakukan tahap feature engineering untuk membentuk matrix segmentasi baru seperti **Product Category, Purchase Size dan Spending Level**.

ini adalah hasil evaluasi distribusi data kami beserta analisa mendalam tentang karakteristik operasional dan profile pelanggan kafe saat ini.

1. Kategori Produk
- **Beverage** berjumlah **3.901** transaksi dengan presentase **51.59%**
- **Food** berjumlah **3.661** transaksi dengan presentase **48.41%**
---
2. Purchase Size
- **Small** berjumlah **3.103** transaksi deengan presentase **41.03%**
- **Medium** berjumlah **3.034** transaksi dengan presentase **40.12%**
- **Large** berjumlah **1.425** transaksi dengan presentase **18.84%**

---

3. Spending level
- **Budget** nerjumlah **2.826** transaksi dengan presentase **37.37%**
- **Standart** berjumlah **2.394** transaksi dengan presentase **31.66%**
- **Premium** berjumlah **1.364** transaksi dengan preseentase **18.04%**
- **Luxury** berjumlah **976** transaksi dengan presentase **12.93%**


**Analisis Evaluaasi**

1. **Karakteristik dari product category**
menurut data, menunjukan kalau perputaran penjualaaan antara makanan dan minuman di kafe berjalan **sangat seimbang**. Kelompok Beverage dan Food hanya memiliki perbandingan sekitar **3.18%**

- insight : Distribusi yang terlihat 50:50 ini membuktikan kalau konsumen tidak hanya menjadikan kafe ini hanya sebagai tempat nongkrong dan minum, tapi juga seebagai destinasi untuk makan. supply chain untuk kedua kategori ini harus di jaga dengan seimbang deengan tingkat kebutuhan yang di sama ratakaan tingginyaa agar tidak terjadi kehilangan potensi penjualan / kehabisan stock.

---
2. **Pola Purchase Size**
Data Volume dari purchase Size menunjukan bahwa adanya pola yang sangat terkonsentrasi. sekitar **81.15%** total transaksi didominasi oleh kombinasi antara Small dan Medium. ini menunjukan adanya ketimpangan yang sangat signifikan yang di dapati jika kita melihat Large yang hanya menyumbang **18.84%**
- insight : karakteristik pengunjung ini bersifat individual dan kelompok kecil (2-3) orang. kafe jarang menerima order masal dalam satu struk. strategi penataan meja sebaiknya dioptimalkan dari kapasitaas meja yang kecil, dan tim pemasaran harus ngerancang paket bundling agar mendorong pelanggan untuk dari yg hanya membeli 1-2 iteem menjadi 3-4 item per transaksi
---
3. **Segmentasi pasar dari Spending Level**
Bisa kita lihat dari sisi finansial kalau rata rata pelanggan kafe ini sangat sensitif terhadap harga, segmen** budget dan standart** menguasai hampir **70%** dari keseluruhan profile konsumen. sedangkan kelompok **premium dan luxury** kalau di gabungkan hanya menyentuh angka **31%**

- insight : Core customer kafe ini adalah pemburu value for money. sebaiknya jangan sampai membuat kebijakan kenaikan harga menu dengan agresif karena sangat beresiko mematikan volume transaksi. fokus utama manajemen adalah harus bergantung pada strategi kuantitas yang tinggi dengan margin rendah / high volume, low margin pada meenu regulr untuk menjaga loyalitas dari pelanggan.


# 9. Business Recomendation

**Recomendation 1 :**  Meeeengoptimalkan Kontributor Revenue Terbesar
melihat Beverage deengan preseentase 51.59% adalah merupakan kontributor transaksi terbeeesar dan utama, kafeee sebaiknya meluncurkan program promosi khusus pada kategori ini.

**Reecomendation 2 :** Diferensiasi Strategi pmasaran
ada perbedaan yg signifikan dari seegmen budget dan prmium dan memerlukan pendekatan yang berbeda beda.
- Budget : fokuskan pada strateegi value for money. andalkan promo "menu hemat" atau misalnya "beli 2 leebih murah" untuk menjaga volume transaksi dari basis pelanggan yg meendominasi ini

- Premium Customer : Fokuskan pada aspek eksklusifitas dan kualitas. tawarkan menu musiman/limiteed edition untuk memaksimalkan margin keuntungan di segmen ini

**Recomendation 3 :** Penyesuaian Layanan
Data menunjukan kalau 81.15% transaksi adalah small dan medium. penyesuaian layanan yang bisa dilakukan aadalah :

- fast Checkout untuk small : kafe menyediakan jalur traansaksi yang cepat bagi konsumen yg hanya membeli 1-2 item untuk meningkatkan perputaaran meja
- bundle promotion : dari yg di lihat pada segmen large  yg ganya 18.84%, berikan pelayanan yg insentif seperti family packet atau diskon khusus untuk pembelian di atas 5 utem untuk mendorong konsumen beralih ke volume belanja yg lebih besar.

In [1]:
!git config --global user.name "Salzabilla Putri"
!git config --global user.email "salzabillahambali@gmail.com"

In [6]:
# Ubah bagian ini:
TOKEN = "YOUR_GITHUB_TOKEN"
USERNAME = "salza-pth"
REPO_NAME = "cafe-sales-analysis"

# Dan ubah baris ini menjadi seperti ini:
!git clone https://{TOKEN}@github.com/{USERNAME}/{REPO_NAME}.git

%cd cafe-sales-analysis

Cloning into 'cafe-sales-analysis'...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 7 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), 151.78 KiB | 1.60 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/cafe-sales-analysis/cafe-sales-analysis


In [5]:
!git add .

!git commit -m "Add notebook and cafe sales dataset"

!git push origin main

[main c7bd0b5] Add notebook and cafe sales dataset
 2 files changed, 15126 insertions(+)
 create mode 100644 cafe_sales_analysis.csv
 create mode 100644 cleaned_cafe_sales.csv
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 150.94 KiB | 705.00 KiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), done.
To https://github.com/salza-pth/cafe-sales-analysis.git
   9aea636..c7bd0b5  main -> main
